# Build BGE-M3 Issue Embedding Index

이 노트북은 GPU runtime에서 `BAAI/bge-m3`로 쟁점 단위 embedding을 만들고 FAISS index를 저장합니다.

로컬에서 할 일은 대부분 repo 스크립트로 처리하고, 이 노트북은 dense embedding 생성만 담당합니다.

## 1. Install dependencies

In [ ]:
!pip -q install sentence-transformers faiss-cpu

## 2. Load issue JSONL

Colab에 repo를 clone한 뒤 `PROJECT_DIR`를 맞추세요. 이미 파일을 업로드했다면 `JSONL_PATH`만 바꾸면 됩니다.

In [ ]:
from pathlib import Path
import json

PROJECT_DIR = Path('/content/sct-project-graphrag')
JSONL_PATH = PROJECT_DIR / 'data/processed/case_issues.jsonl'
OUT_DIR = PROJECT_DIR / 'data/indexes'
OUT_DIR.mkdir(parents=True, exist_ok=True)

rows = [json.loads(line) for line in JSONL_PATH.open(encoding='utf-8') if line.strip()]
len(rows), rows[0].keys()

## 3. Convert documents to issue-level records

In [ ]:
def build_issue_text(record):
    parts = []
    if record.get('tax_item'):
        parts.append(f"[세목] {record['tax_item']}")
    if record.get('case_title'):
        parts.append(f"[판시사항] {record['case_title']}")
    if record.get('issue'):
        parts.append(f"[쟁점] {record['issue']}")
    if record.get('taxpayer_argument'):
        parts.append(f"[납세자 주장] {record['taxpayer_argument']}")
    return '\n'.join(parts)

records = []
texts = []
for row in rows:
    for issue_index, issue in enumerate(row.get('issues', []) or []):
        record = {
            'document_id': row.get('document_id', ''),
            'issue_index': issue_index,
            'case_title': row.get('case_title', ''),
            'decision_type': row.get('decision_type', ''),
            'tax_item': row.get('tax_item', ''),
            'issue': issue.get('issue', ''),
            'taxpayer_argument': issue.get('taxpayer_argument', ''),
            'judgment_reasoning': issue.get('judgment_reasoning', ''),
            'conclusion': issue.get('conclusion', ''),
        }
        record['text'] = build_issue_text(record)
        records.append(record)
        texts.append(record['text'])

len(records), texts[0][:300]

## 4. Build BGE-M3 embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

MODEL_NAME = 'BAAI/bge-m3'
model = SentenceTransformer(MODEL_NAME)
emb = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype('float32')

emb.shape

## 5. Save FAISS index and metadata

In [ ]:
import faiss
import pickle

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

faiss.write_index(index, str(OUT_DIR / 'bge_m3_issue_index.faiss'))
(OUT_DIR / 'bge_m3_issue_metadata.pkl').write_bytes(pickle.dumps(records))

print('vectors:', index.ntotal)
print('metadata:', len(records))
print('saved to:', OUT_DIR)

## 6. Smoke test

In [ ]:
query = '사업자등록 전 매입세액을 공제받을 수 있는지'
q = model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
scores, idxs = index.search(q, 5)

for rank, (score, idx) in enumerate(zip(scores[0], idxs[0]), start=1):
    r = records[idx]
    print(rank, round(float(score), 3), r['document_id'])
    print('쟁점:', r['issue'])
    print()